# RSNA Knee Abnormality Detection — Label Audit v2 (Robust)

Notebook này thay cho `Label Audit v1`.

## Mục tiêu
Audit trực tiếp các **dataset pseudo/LLM labels của TRAIN** so với 58 gold studies.

### Điểm sửa so với v1
- Chỉ scan **direct datasets** trong `/kaggle/input/datasets/...`
- Bỏ qua toàn bộ notebook outputs trong `/kaggle/input/notebooks/...`
- Bỏ qua mọi file tên `submission*.csv`
- Có bảng `accepted_sources` và `rejected_files` rõ ràng
- `metrics` luôn có schema cố định → không còn `KeyError: 'label'`
- Nếu không tìm được source hợp lệ, notebook dừng bằng diagnostic rõ ràng

## Input cần attach
Bắt buộc: competition **RSNA Knee Abnormality Detection**.

Nên attach trực tiếp dưới **DATASETS**:
- `RSNA Knee LLM Report Labels`
- `RSNA Knee LLM-read report labels`
- `RSNA Knee LLM Report Labels (GPT-5.6-Sol)` nếu tìm được

Có thể attach thêm:
- `RSNA Knee hybrid report labels`
- `RSNA Knee LLM Labels — 4-Source Merged`

GPU: **không cần**.

In [ ]:
from pathlib import Path
import os, re, json, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score
warnings.filterwarnings("ignore")
SEED=42
BOOTSTRAP_ITERS=500
LABELS=["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
print("Ready")

In [ ]:
def find_competition_root():
    roots=[]
    for p in Path('/kaggle/input').rglob('train.csv'):
        try:
            df=pd.read_csv(p,nrows=2)
            if {'StudyInstanceUID','ACL','MCL','Report'}.issubset(df.columns): roots.append(p.parent)
        except: pass
    if not roots: raise FileNotFoundError('Attach competition input')
    comp=[x for x in roots if '/competitions/' in str(x)]
    return comp[0] if comp else roots[0]
COMP_ROOT=find_competition_root(); train=pd.read_csv(COMP_ROOT/'train.csv')
train['n_available_labels']=train[LABELS].notna().sum(axis=1)
gold=train.loc[train['n_available_labels'].eq(len(LABELS)),['StudyInstanceUID']+LABELS].copy()
gold['StudyInstanceUID']=gold['StudyInstanceUID'].astype(str)
train_uid_set=set(train['StudyInstanceUID'].astype(str)); gold_uid_set=set(gold['StudyInstanceUID'])
print('Competition root:',COMP_ROOT); print('Train:',len(train),'Gold:',len(gold))

In [ ]:
def norm_name(x):
    x=str(x).strip().lower().replace("'",''); x=re.sub(r'[^a-z0-9]+',' ',x); return re.sub(r'\s+',' ',x).strip()
ALIASES={
'ACL':['acl','acl injury','acl tear','anterior cruciate ligament'],
'MCL':['mcl','mcl injury','mcl tear','medial collateral ligament'],
'Medial Meniscus':['medial meniscus','medial meniscus tear','medial meniscal tear','medial meniscal','mm'],
'Lateral Meniscus':['lateral meniscus','lateral meniscus tear','lateral meniscal tear','lateral meniscal','lm'],
'Medial OA':['medial oa','medial osteoarthritis','medial tibiofemoral oa','medial tibiofemoral osteoarthritis'],
'Lateral OA':['lateral oa','lateral osteoarthritis','lateral tibiofemoral oa','lateral tibiofemoral osteoarthritis'],
'PF OA':['pf oa','patellofemoral oa','patellofemoral osteoarthritis','patello femoral oa'],
'Effusion':['effusion','joint effusion'],'Synovitis':['synovitis'],
"Baker's":['bakers','bakers cyst','baker cyst','baker s cyst','popliteal cyst'],
'Contusion':['contusion','bone contusion','bone bruise','bone bruising'],'Fracture':['fracture','bone fracture']}
ALIAS_TO_LABEL={}
for label,arr in ALIASES.items():
    ALIAS_TO_LABEL[norm_name(label)]=label
    for a in arr: ALIAS_TO_LABEL[norm_name(a)]=label
UID_KEYS={norm_name(x) for x in ['StudyInstanceUID','study instance uid','study uid','studyid','study id','uid']}
def infer_uid_col(df):
    mp={norm_name(c):c for c in df.columns}
    for k in UID_KEYS:
        if k in mp:return mp[k]
    return None
def infer_label_cols(df):
    out={}
    for c in df.columns:
        nc=norm_name(c)
        if nc in ALIAS_TO_LABEL: out[ALIAS_TO_LABEL[nc]]=c; continue
        for t in LABELS:
            nt=norm_name(t)
            if nc in {f'{nt} score',f'{nt} prob',f'{nt} probability',f'{nt} label',f'{nt} pred',f'{nt} prediction',f'pred {nt}',f'prob {nt}',f'score {nt}'}:
                out[t]=c; break
    return out

In [ ]:
def safe_read(path,nrows=None):
    path=Path(path)
    try:
        if path.suffix.lower()=='.csv': return pd.read_csv(path,nrows=nrows)
        if path.suffix.lower() in {'.parquet','.pq'}:
            df=pd.read_parquet(path); return df.head(nrows) if nrows else df
        if path.suffix.lower()=='.json':
            try: df=pd.read_json(path)
            except: df=pd.read_json(path,lines=True)
            return df.head(nrows) if nrows else df
    except: return None

def coerce_score(s):
    num=pd.to_numeric(s,errors='coerce')
    if num.notna().mean()>=0.70:return num.astype(float)
    mp={'positive':1.,'pos':1.,'present':1.,'yes':1.,'true':1.,'negative':0.,'neg':0.,'absent':0.,'no':0.,'false':0.,'uncertain':.5,'indeterminate':.5,'possible':.5,'unknown':np.nan,'unmentioned':np.nan,'not mentioned':np.nan}
    return s.astype(str).str.strip().str.lower().map(mp).astype(float)

In [ ]:
all_paths=[]
root=Path('/kaggle/input/datasets')
if root.exists():
    for ext in ['*.csv','*.parquet','*.pq','*.json']: all_paths.extend(root.rglob(ext))
else:
    for ext in ['*.csv','*.parquet','*.pq','*.json']:
        for p in Path('/kaggle/input').rglob(ext):
            sp=str(p)
            if '/notebooks/' in sp or '/competitions/' in sp: continue
            all_paths.append(p)
accepted=[]; rejected=[]
for p in sorted(set(all_paths),key=str):
    if 'submission' in p.name.lower(): rejected.append({'path':str(p),'reason':'submission/test prediction'}); continue
    prev=safe_read(p,20)
    if prev is None or len(prev)==0: rejected.append({'path':str(p),'reason':'unreadable/empty'}); continue
    uid=infer_uid_col(prev); labs=infer_label_cols(prev)
    if uid is None: rejected.append({'path':str(p),'reason':'no UID'}); continue
    if not labs: rejected.append({'path':str(p),'reason':'no target cols'}); continue
    full=safe_read(p)
    if full is None: rejected.append({'path':str(p),'reason':'full read failed'}); continue
    uid=infer_uid_col(full); labs=infer_label_cols(full)
    uids=set(full[uid].astype(str)); tr=len(uids & train_uid_set); go=len(uids & gold_uid_set)
    if tr==0: rejected.append({'path':str(p),'reason':'0 train overlap'}); continue
    accepted.append({'path':str(p),'dataset_folder':p.parent.name,'file_name':p.name,'n_rows':len(full),'uid_col':uid,'n_mapped_labels':len(labs),'mapped_labels':', '.join(labs.keys()),'train_uid_overlap':tr,'gold_uid_overlap':go})
accepted_sources=pd.DataFrame(accepted); rejected_files=pd.DataFrame(rejected)
if len(accepted_sources): accepted_sources=accepted_sources.sort_values(['train_uid_overlap','n_mapped_labels','n_rows'],ascending=False).reset_index(drop=True)
print('DIRECT candidate sources:',len(accepted_sources)); display(accepted_sources)
print('Rejected first 50'); display(rejected_files.head(50))
if len(accepted_sources)==0: raise RuntimeError('No direct pseudo-label dataset found. Attach DATASETS, not only NOTEBOOKS.')

In [ ]:
sources={}; meta=[]
for _,row in accepted_sources.iterrows():
    raw=safe_read(row['path']); uid=infer_uid_col(raw); labs=infer_label_cols(raw)
    norm=pd.DataFrame({'StudyInstanceUID':raw[uid].astype(str)})
    for label,col in labs.items(): norm[label]=coerce_score(raw[col])
    norm=norm.groupby('StudyInstanceUID',as_index=False).mean(numeric_only=True)
    sid=f'S{len(sources)+1:02d}'; sources[sid]=norm
    vals=norm[[c for c in LABELS if c in norm.columns]].stack().dropna()
    meta.append({'source_id':sid,'dataset_folder':row['dataset_folder'],'file_name':row['file_name'],'path':row['path'],'n_rows':len(norm),'n_labels':len([c for c in LABELS if c in norm.columns]),'train_uid_overlap':row['train_uid_overlap'],'gold_uid_overlap':row['gold_uid_overlap'],'binary_fraction':float(vals.isin([0,1]).mean()) if len(vals) else np.nan})
source_manifest=pd.DataFrame(meta); display(source_manifest)

In [ ]:
def safe_auc(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); m=np.isfinite(y)&np.isfinite(p); y,p=y[m],p[m]
    return np.nan if len(y)<3 or len(np.unique(y))<2 else float(roc_auc_score(y,p))
def safe_ap(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); m=np.isfinite(y)&np.isfinite(p); y,p=y[m],p[m]
    return np.nan if len(y)<3 or np.sum(y==1)==0 else float(average_precision_score(y,p))
def boot_ci(y,p,n=BOOTSTRAP_ITERS):
    y=np.asarray(y,float); p=np.asarray(p,float); m=np.isfinite(y)&np.isfinite(p); y,p=y[m],p[m]
    if len(y)<3 or len(np.unique(y))<2:return np.nan,np.nan
    rng=np.random.default_rng(SEED); vals=[]
    for _ in range(n):
        idx=rng.integers(0,len(y),len(y)); yy,pp=y[idx],p[idx]
        if len(np.unique(yy))<2: continue
        vals.append(roc_auc_score(yy,pp))
    return (np.nan,np.nan) if len(vals)<20 else (float(np.percentile(vals,2.5)),float(np.percentile(vals,97.5)))

In [ ]:
METRIC_COLUMNS=['source_id','label','n_gold_overlap','n_positive','n_negative','coverage','auc','auc_ci_low','auc_ci_high','ap','accuracy_05','balanced_accuracy_05','precision_05','recall_05','f1_05']
rows=[]
for sid,sdf in sources.items():
    m=gold.merge(sdf,on='StudyInstanceUID',how='left',suffixes=('_gold','_pred'))
    for label in LABELS:
        gc,pc=label+'_gold',label+'_pred'
        if pc not in m.columns: continue
        y=pd.to_numeric(m[gc],errors='coerce').to_numpy(float); p=pd.to_numeric(m[pc],errors='coerce').to_numpy(float)
        mask=np.isfinite(y)&np.isfinite(p); yy,pp=y[mask],p[mask]
        if len(yy)==0: continue
        auc=safe_auc(yy,pp); ap=safe_ap(yy,pp); lo,hi=boot_ci(yy,pp); hard=(pp>=.5).astype(int)
        rows.append({'source_id':sid,'label':label,'n_gold_overlap':len(yy),'n_positive':int((yy==1).sum()),'n_negative':int((yy==0).sum()),'coverage':len(yy)/len(gold),'auc':auc,'auc_ci_low':lo,'auc_ci_high':hi,'ap':ap,'accuracy_05':accuracy_score(yy,hard),'balanced_accuracy_05':balanced_accuracy_score(yy,hard) if len(np.unique(yy))==2 else np.nan,'precision_05':precision_score(yy,hard,zero_division=0),'recall_05':recall_score(yy,hard,zero_division=0),'f1_05':f1_score(yy,hard,zero_division=0)})
metrics=pd.DataFrame(rows,columns=METRIC_COLUMNS)
print('Per-label gold audit')
if metrics.empty: raise RuntimeError('No usable GOLD overlap metrics; inspect source_manifest.')
display(metrics.sort_values(['label','auc'],ascending=[True,False]))

In [ ]:
macro=(metrics.groupby('source_id').agg(macro_auc=('auc','mean'),macro_ap=('ap','mean'),mean_coverage=('coverage','mean'),labels_evaluated=('label','nunique')).reset_index().merge(source_manifest,on='source_id',how='left').sort_values('macro_auc',ascending=False))
display(macro)
pivot=metrics.pivot(index='label',columns='source_id',values='auc'); display(pivot.reindex(LABELS).round(3))

In [ ]:
source_ids=list(sources); pair=[]
for i in range(len(source_ids)):
    for j in range(i+1,len(source_ids)):
        a,b=source_ids[i],source_ids[j]; m=sources[a].merge(sources[b],on='StudyInstanceUID',how='inner',suffixes=('_a','_b'))
        for label in LABELS:
            ca,cb=label+'_a',label+'_b'
            if ca not in m.columns or cb not in m.columns: continue
            x=pd.to_numeric(m[ca],errors='coerce').to_numpy(float); y=pd.to_numeric(m[cb],errors='coerce').to_numpy(float); v=np.isfinite(x)&np.isfinite(y); x,y=x[v],y[v]
            if len(x): pair.append({'source_a':a,'source_b':b,'label':label,'n_overlap':len(x),'hard_disagreement_rate':float(np.mean((x>=.5)!=(y>=.5))),'mean_abs_score_difference':float(np.mean(np.abs(x-y)))})
pairwise=pd.DataFrame(pair,columns=['source_a','source_b','label','n_overlap','hard_disagreement_rate','mean_abs_score_difference'])
if len(pairwise): display(pairwise.sort_values(['hard_disagreement_rate','mean_abs_score_difference'],ascending=False).head(100))
else: print('Need >=2 usable sources for disagreement audit')

In [ ]:
all_uids=train[['StudyInstanceUID']].copy(); all_uids['StudyInstanceUID']=all_uids['StudyInstanceUID'].astype(str); cube=all_uids.copy()
for sid,sdf in sources.items():
    ren={label:f'{sid}__{label}' for label in LABELS if label in sdf.columns}; cube=cube.merge(sdf.rename(columns=ren),on='StudyInstanceUID',how='left')
silver_mean=all_uids.copy(); silver_median=all_uids.copy(); silver_best=all_uids.copy(); best={}
for label in LABELS:
    cols=[f'{sid}__{label}' for sid in source_ids if f'{sid}__{label}' in cube.columns]
    if not cols: silver_mean[label]=np.nan; silver_median[label]=np.nan; silver_best[label]=np.nan; continue
    arr=cube[cols].to_numpy(float); silver_mean[label]=np.nanmean(arr,axis=1); silver_median[label]=np.nanmedian(arr,axis=1)
    mm=metrics[metrics.label.eq(label)].sort_values(['auc','coverage','n_gold_overlap'],ascending=False)
    if len(mm):
        sid=mm.iloc[0].source_id; best[label]=sid; silver_best[label]=cube[f'{sid}__{label}']
    else: silver_best[label]=silver_mean[label]
gids=set(gold.StudyInstanceUID)
rm=lambda df: df[~df.StudyInstanceUID.isin(gids)].reset_index(drop=True)
silver_mean,silver_median,silver_best=rm(silver_mean),rm(silver_median),rm(silver_best)
print('Best source per target:',best); print('Weak rows:',len(silver_best))

In [ ]:
OUT=Path('/kaggle/working/rsna_label_audit_v2'); OUT.mkdir(parents=True,exist_ok=True)
accepted_sources.to_csv(OUT/'accepted_sources.csv',index=False); rejected_files.to_csv(OUT/'rejected_files.csv',index=False); source_manifest.to_csv(OUT/'source_manifest.csv',index=False); metrics.to_csv(OUT/'gold_per_label_metrics.csv',index=False); macro.to_csv(OUT/'gold_macro_summary.csv',index=False); pairwise.to_csv(OUT/'pairwise_disagreement.csv',index=False); silver_mean.to_csv(OUT/'silver_mean_consensus.csv',index=False); silver_median.to_csv(OUT/'silver_median_consensus.csv',index=False); silver_best.to_csv(OUT/'silver_best_source_per_target.csv',index=False)
lines=['RSNA KNEE LABEL AUDIT v2','='*70,f'Train studies: {len(train)}',f'Gold studies: {len(gold)}',f'Accepted direct pseudo-label files: {len(accepted_sources)}','', 'SOURCES']
for _,r in source_manifest.iterrows(): lines.append(f"{r.source_id} | {r.dataset_folder}/{r.file_name} | rows={r.n_rows} labels={r.n_labels} train_overlap={r.train_uid_overlap} gold_overlap={r.gold_uid_overlap}")
lines+=['','MACRO GOLD AUDIT']
for _,r in macro.iterrows(): lines.append(f"{r.source_id} | macro_auc={r.macro_auc:.4f} macro_ap={r.macro_ap:.4f} coverage={r.mean_coverage:.4f}")
lines+=['','BEST SOURCE PER TARGET']
for label in LABELS:
    mm=metrics[metrics.label.eq(label)].sort_values(['auc','coverage'],ascending=False)
    if len(mm):
        r=mm.iloc[0]; lines.append(f"{label}: {r.source_id} | AUC={r.auc:.4f} CI=[{r.auc_ci_low:.4f},{r.auc_ci_high:.4f}] n={int(r.n_gold_overlap)}")
result='\n'.join(lines); (OUT/'RESULTS_TO_SEND_BACK.txt').write_text(result,encoding='utf-8'); print(result)
ZIP=Path('/kaggle/working/RSNA_Knee_Label_Audit_v2_Artifacts.zip')
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file(): z.write(p,arcname=p.relative_to(OUT))
print('ZIP:',ZIP)